# 🐦 Twitter Sentiment Analysis using NLP
### Coding Samurai Internship - Data Science Project (Level 3)
---
**Model:** Logistic Regression + NLP  
**Dataset:** Twitter Entity Sentiment Analysis (74,000+ tweets)  
**Labels:** Positive | Negative | Neutral | Irrelevant

## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk textblob wordcloud

## 📚 Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('✅ All libraries imported successfully!')

## 📥 Step 3: Load Dataset

In [ ]:
# Load training and validation data
# Make sure twitter_training.csv and twitter_validation.csv are in same folder
train_df = pd.read_csv('twitter_training.csv', header=None,
                        names=['ID', 'Topic', 'Sentiment', 'Tweet'])
val_df   = pd.read_csv('twitter_validation.csv', header=None,
                        names=['ID', 'Topic', 'Sentiment', 'Tweet'])

print(f'✅ Data Loaded!')
print(f'Training samples   : {len(train_df)}')
print(f'Validation samples : {len(val_df)}')
train_df.head(10)

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(train_df.info())
print('\n=== Missing Values ===')
print(train_df.isnull().sum())
print('\n=== Sentiment Distribution ===')
print(train_df['Sentiment'].value_counts())

In [ ]:
# Plot Sentiment Distribution
plt.figure(figsize=(10, 5))

# Bar chart
plt.subplot(1, 2, 1)
colors = ['#2ecc71', '#e74c3c', '#3498db', '#95a5a6']
train_df['Sentiment'].value_counts().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Sentiment Distribution (Count)', fontsize=13, fontweight='bold')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Pie chart
plt.subplot(1, 2, 2)
train_df['Sentiment'].value_counts().plot(kind='pie', colors=colors,
                                           autopct='%1.1f%%', startangle=90)
plt.title('Sentiment Distribution (%)', fontsize=13, fontweight='bold')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Top Topics
plt.figure(figsize=(12, 5))
top_topics = train_df['Topic'].value_counts().head(15)
sns.barplot(x=top_topics.values, y=top_topics.index, palette='viridis')
plt.title('Top 15 Topics in Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Topic')
plt.tight_layout()
plt.show()

In [ ]:
# Tweet length analysis
train_df['Tweet_Length'] = train_df['Tweet'].astype(str).apply(len)

plt.figure(figsize=(12, 5))
for sentiment, color in zip(['Positive', 'Negative', 'Neutral', 'Irrelevant'],
                             ['green', 'red', 'blue', 'gray']):
    subset = train_df[train_df['Sentiment'] == sentiment]['Tweet_Length']
    subset.plot(kind='hist', alpha=0.5, color=color, label=sentiment, bins=50)

plt.title('Tweet Length Distribution by Sentiment', fontsize=14, fontweight='bold')
plt.xlabel('Tweet Length (characters)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

## 🧹 Step 5: Text Cleaning & Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                          # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)        # Remove URLs
    text = re.sub(r'@\w+', '', text)                  # Remove @mentions
    text = re.sub(r'#\w+', '', text)                  # Remove hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)           # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()          # Remove extra spaces
    # Remove stopwords and lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

# Apply cleaning
train_df['Cleaned_Tweet'] = train_df['Tweet'].apply(clean_text)
val_df['Cleaned_Tweet']   = val_df['Tweet'].apply(clean_text)

print('✅ Text cleaning done!')
print('\nOriginal  :', train_df['Tweet'].iloc[0])
print('Cleaned   :', train_df['Cleaned_Tweet'].iloc[0])

In [ ]:
# WordCloud for Positive tweets
positive_text = ' '.join(train_df[train_df['Sentiment'] == 'Positive']['Cleaned_Tweet'])

wordcloud = WordCloud(width=800, height=400, background_color='white',
                      colormap='Greens', max_words=100).generate(positive_text)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('😊 Positive Tweets - WordCloud', fontsize=13, fontweight='bold')
plt.axis('off')

# WordCloud for Negative tweets
negative_text = ' '.join(train_df[train_df['Sentiment'] == 'Negative']['Cleaned_Tweet'])
wordcloud2 = WordCloud(width=800, height=400, background_color='white',
                       colormap='Reds', max_words=100).generate(negative_text)

plt.subplot(1, 2, 2)
plt.imshow(wordcloud2, interpolation='bilinear')
plt.title('😠 Negative Tweets - WordCloud', fontsize=13, fontweight='bold')
plt.axis('off')

plt.tight_layout()
plt.show()

## ⚙️ Step 6: Feature Engineering - TF-IDF

In [ ]:
# Remove 'Irrelevant' tweets — focus on 3 sentiments
train_filtered = train_df[train_df['Sentiment'] != 'Irrelevant'].copy()
val_filtered   = val_df[val_df['Sentiment'] != 'Irrelevant'].copy()

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(train_filtered['Cleaned_Tweet'])
X_test  = tfidf.transform(val_filtered['Cleaned_Tweet'])

y_train = train_filtered['Sentiment']
y_test  = val_filtered['Sentiment']

print(f'✅ TF-IDF done!')
print(f'Training features : {X_train.shape}')
print(f'Testing features  : {X_test.shape}')

## 🤖 Step 7: Train Model - Logistic Regression

In [ ]:
# Train Logistic Regression
print('🚀 Training model...')
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print('✅ Model trained!')

## 📊 Step 8: Evaluate Model

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'\n✅ Model Accuracy: {accuracy*100:.2f}%')
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Negative', 'Neutral'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Negative', 'Neutral'],
            yticklabels=['Positive', 'Negative', 'Neutral'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

## 🔮 Step 9: TextBlob Sentiment Analysis (Comparison)

In [ ]:
# TextBlob Analysis on sample tweets
def get_textblob_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    if score > 0.1:
        return 'Positive'
    elif score < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

# Apply on validation data
val_filtered = val_filtered.copy()
val_filtered['TextBlob_Sentiment'] = val_filtered['Tweet'].apply(get_textblob_sentiment)

tb_accuracy = accuracy_score(
    val_filtered[val_filtered['Sentiment'] != 'Irrelevant']['Sentiment'],
    val_filtered[val_filtered['Sentiment'] != 'Irrelevant']['TextBlob_Sentiment']
)

print('=== Model Comparison ===')
print(f'Logistic Regression Accuracy : {accuracy*100:.2f}%')
print(f'TextBlob Accuracy            : {tb_accuracy*100:.2f}%')

## 🧪 Step 10: Test on Custom Tweets

In [ ]:
# Test on your own tweets!
custom_tweets = [
    "I love this product! It is absolutely amazing!",
    "This is the worst experience I have ever had.",
    "The weather is okay today, nothing special.",
    "TCS quarterly results exceeded all expectations!",
    "Terrible customer service, very disappointed."
]

# Clean and predict
cleaned = [clean_text(t) for t in custom_tweets]
vectorized = tfidf.transform(cleaned)
predictions = model.predict(vectorized)

print('=== Custom Tweet Predictions ===')
print('-' * 60)
for tweet, pred in zip(custom_tweets, predictions):
    emoji = '😊' if pred == 'Positive' else '😠' if pred == 'Negative' else '😐'
    print(f'{emoji} [{pred:8}] {tweet[:50]}...' if len(tweet) > 50 else f'{emoji} [{pred:8}] {tweet}')
print('-' * 60)

## ✅ Step 11: Summary & Conclusion

In [ ]:
print('='*55)
print('   📋 PROJECT SUMMARY - Twitter Sentiment Analysis')
print('='*55)
print(f'  Dataset          : Twitter Entity Sentiment Analysis')
print(f'  Total Tweets     : {len(train_df):,}')
print(f'  Sentiments       : Positive, Negative, Neutral')
print(f'  Text Cleaning    : URLs, mentions, stopwords removed')
print(f'  Features         : TF-IDF (10,000 features, bigrams)')
print(f'  Model            : Logistic Regression')
print('-'*55)
print(f'  Model Accuracy   : {accuracy*100:.2f}%')
print(f'  TextBlob (base)  : {tb_accuracy*100:.2f}%')
print('='*55)
print('\n  Key Learnings:')
print('  1. NLP text preprocessing techniques')
print('  2. TF-IDF vectorization')
print('  3. Sentiment classification using ML')
print('  4. WordCloud visualization')
print('  5. TextBlob vs ML model comparison')

---
## 📌 Key Learnings

1. **Data Loading** — Real Twitter dataset with 74,000+ tweets
2. **EDA** — Sentiment distribution, topic analysis, tweet length
3. **Text Cleaning** — URLs, mentions, hashtags, stopwords removed
4. **WordCloud** — Visual representation of frequent words
5. **TF-IDF** — Converted text to numerical features
6. **Logistic Regression** — Trained sentiment classifier
7. **TextBlob** — Rule-based sentiment comparison
8. **Custom Predictions** — Tested on real custom tweets

---
*Project by: [Your Name] | Coding Samurai Data Science Internship*